# Experiment 12 — ProtoSSM v4 + Improved MLP Probes

**Source:** Adaptation of `discussion/pantanal-distill-birdclef2026-improvement.ipynb` (public LB ~0.924+).

**Key upgrades over exp11:**
- ProtoSSM v4: `d_model=256`, 3 SSM layers, `TemporalCrossAttention` with FFN
- Focal loss (γ=2.5) + label smoothing + knowledge distillation
- Mixup augmentation (α=0.4) after warmup
- SWA (Stochastic Weight Averaging, starts at 65% of training)
- Taxonomic auxiliary head (class_name groups)
- MLP probes: `pca_dim=128`, hidden=(256,128), richer sequential features
- Rank-aware scaling (power=0.4)

In [1]:
import os, re, gc, time, warnings, concurrent.futures
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

torch.manual_seed(42)
np.random.seed(42)
DEVICE = 'cpu'

PROJECT_ROOT = Path('/Users/jjannik/Development/kaggle/birdclef')
BASE      = PROJECT_ROOT / 'data'
MODEL_DIR = BASE / 'models' / 'perch_tf'
ONNX_PATH = BASE / 'models' / 'perch' / 'perch_v2_no_dft.onnx'
CACHE_DIR = BASE / 'cache'

# Reuse exp11 cache
CACHE_META = CACHE_DIR / 'exp11_perch_meta.csv'
CACHE_NPZ  = CACHE_DIR / 'exp11_perch_arrays.npz'

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12
N_OOF_SPLITS   = 3

CFG = {
    'proto_ssm': {
        'd_model': 256, 'd_state': 16, 'n_ssm_layers': 3,
        'dropout': 0.12, 'n_sites': 20, 'meta_dim': 24,
        'use_cross_attn': True, 'cross_attn_heads': 8,
    },
    'proto_ssm_train': {
        'n_epochs': 20, 'lr': 8e-4, 'weight_decay': 1e-3,
        'patience': 8, 'pos_weight_cap': 25.0,
        'distill_weight': 0.15, 'label_smoothing': 0.03,
        'mixup_alpha': 0.4, 'focal_gamma': 2.5,
        'swa_start_frac': 0.65,
    },
    'mlp': {'pca_dim': 128, 'hidden': (256, 128), 'min_pos': 5, 'alpha_blend': 0.25},
    'rank_aware_power': 0.4,
    'delta_smooth_alpha': 0.20,
    'lambda_prior': 0.10,
}

_WALL_START = time.time()
print('Setup done')

Setup done


In [2]:
# Data loading
taxonomy          = pd.read_csv(BASE / 'taxonomy.csv')
sample_sub        = pd.read_csv(BASE / 'sample_submission.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': 'unknown', 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(';'):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc['end_sec'] = pd.to_timedelta(sc['end']).dt.total_seconds().astype(int)
sc['row_id'] = sc['filename'].str.replace('.ogg', '', regex=False) + '_' + sc['end_sec'].astype(str)
_meta = sc['filename'].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc['label_list']):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby('filename').size()
LABEL_WINDOWS = int(windows_per_file.mode().iloc[0])
full_files = sorted(windows_per_file[windows_per_file == LABEL_WINDOWS].index.tolist())
sc['fully_labeled'] = sc['filename'].isin(full_files)
full_rows = sc[sc['fully_labeled']].sort_values(['filename', 'end_sec']).reset_index(drop=False)
Y_FULL = Y_SC[full_rows['index'].to_numpy()]

print(f'Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}')
print(f'Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0)>0).sum())}')

Classes: 234 | Fully-labeled files: 59
Full-file windows: 708 | Active classes: 71


In [3]:
# Load Perch ONNX + label mapping
import re as _re

_so = ort.SessionOptions(); _so.intra_op_num_threads = 4
ONNX_SESSION    = ort.InferenceSession(str(ONNX_PATH), sess_options=_so, providers=['CPUExecutionProvider'])
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}

bc_labels = (pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv')
             .reset_index().rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'}))
NO_LABEL  = len(bc_labels)
mapping   = taxonomy.merge(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc    = mapping.set_index('primary_label')['bc_index']

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
proxy_map    = {}
for _, row in taxonomy[taxonomy['primary_label'].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits  = bc_labels[bc_labels['scientific_name'].astype(str).str.match(rf'^{_re.escape(genus)}\s', na=False)]
    if len(hits) > 0:
        proxy_map[label_to_idx[row['primary_label']]] = hits['bc_index'].astype(int).tolist()
proxy_map = {k: v for k, v in proxy_map.items() if CLASS_NAME_MAP.get(PRIMARY_LABELS[k]) in {'Amphibia','Insecta','Aves'}}

print(f'Mapped: {MAPPED_MASK.sum()} / {N_CLASSES}  proxies: {len(proxy_map)}')

Mapped: 203 / 234  proxies: 3

In [4]:
# Perch inference engine (only needed if no cache)
def read_60s(path):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=8, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids = np.empty(n_rows, dtype=object); filenames = np.empty(n_rows, dtype=object)
    sites   = np.empty(n_rows, dtype=object); hours = np.zeros(n_rows, dtype=np.int16)
    scores  = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs    = np.zeros((n_rows, 1536), dtype=np.float32)
    wr = 0
    itr = tqdm(range(0, len(paths), batch_files), desc='Perch') if verbose else range(0, len(paths), batch_files)
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as exe:
        nxt_p  = paths[:batch_files]
        nxt_f  = [exe.submit(read_60s, p) for p in nxt_p]
        for start in itr:
            bp = nxt_p; ba = [f.result() for f in nxt_f]; br = wr
            ns = start + batch_files
            if ns < len(paths):
                nxt_p = paths[ns:ns+batch_files]; nxt_f = [exe.submit(read_60s, p) for p in nxt_p]
            x = np.empty((len(bp)*N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            for bi, path in enumerate(bp):
                y = ba[bi]; meta = parse_fname(path.name); stem = path.stem
                x[bi*N_WINDOWS:(bi+1)*N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr+N_WINDOWS] = [f'{stem}_{t}' for t in range(5,65,5)]
                filenames[wr:wr+N_WINDOWS] = path.name
                sites    [wr:wr+N_WINDOWS] = meta['site']
                hours    [wr:wr+N_WINDOWS] = meta['hour_utc']
                wr += N_WINDOWS
            outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP['label']].astype(np.float32)
            emb    = outs[ONNX_OUT_MAP['embedding']].astype(np.float32)
            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb
            for pos_idx, bc_idxs in proxy_map.items():
                scores[br:wr, pos_idx] = logits[:, np.array(bc_idxs, dtype=np.int32)].max(axis=1)
            del x, logits, emb, ba; gc.collect()
    return pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours}), scores, embs

print('Perch engine ready')

Perch engine ready


In [5]:
# Load cached Perch features (built in exp11)
def _pick(arr, keys, ncols):
    for k in keys:
        if k in arr.files: return arr[k]
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == ncols: return v
    raise KeyError(f'{keys} not in {arr.files}')

if not (CACHE_META.exists() and CACHE_NPZ.exists()):
    print('No cache found — building...')
    train_paths = [BASE / 'train_soundscapes' / fn for fn in full_files]
    t0 = time.time()
    meta_b, sc_b, emb_b = run_perch(train_paths, batch_files=8, verbose=True)
    print(f'  Done in {time.time()-t0:.1f}s')
    meta_b.to_csv(CACHE_META, index=False)
    np.savez(CACHE_NPZ, scores=sc_b.astype(np.float32), embs=emb_b.astype(np.float32),
             primary_labels=np.array(PRIMARY_LABELS))
else:
    print('Loading exp11 Perch cache...')

meta_tr = pd.read_csv(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr   = _pick(_arr, ['scores','sc','logits','arr_0'], N_CLASSES).astype(np.float32)
emb_tr  = _pick(_arr, ['embs','emb','embeddings','arr_1'], 1536).astype(np.float32)

row_id_to_index = full_rows.set_index('row_id')['index']
Y_FULL_aligned  = Y_SC[row_id_to_index.loc[meta_tr['row_id']].to_numpy()]

print(f'sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y: {Y_FULL_aligned.shape}')

Loading exp11 Perch cache...
sc_tr: (708, 234)  emb_tr: (708, 1536)  Y: (708, 234)


In [6]:
# Metrics + loss
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def focal_bce(logits, targets, gamma=2.5, pos_weight=None):
    bce = F.binary_cross_entropy_with_logits(logits, targets,
              pos_weight=pos_weight, reduction='none')
    p_t = torch.exp(-bce)
    return ((1 - p_t) ** gamma * bce).mean()

print('Metrics + loss ready')

Metrics + loss ready


In [7]:
# Post-processing helpers

def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)
    site_keys = sorted(sc_df['site'].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p    = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]; mask = sc_df['site'].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_labels[mask].mean(axis=0)
    hour_keys = sorted(sc_df['hour_utc'].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p    = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]; mask = sc_df['hour_utc'].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_labels[mask].mean(axis=0)
    return {'global_p': global_p,
            'site_to_i': site_to_i, 'site_p': site_p, 'site_n': site_n,
            'hour_to_i': hour_to_i, 'hour_p': hour_p, 'hour_n': hour_n}

def apply_prior(scores, sites, hours, tables, lam=0.10):
    eps = 1e-4; out = scores.copy()
    p = np.tile(tables['global_p'], (len(scores), 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables['hour_to_i']:
            j  = tables['hour_to_i'][h]; nh = tables['hour_n'][j]
            w  = nh / (nh + 8.0)
            p[i] = w * tables['hour_p'][j] + (1-w) * tables['global_p']
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables['site_to_i']:
            j  = tables['site_to_i'][s]; ns = tables['site_n'][j]
            w  = ns / (ns + 8.0)
            p[i] = w * tables['site_p'][j] + (1-w) * p[i]
    p    = np.clip(p, eps, 1-eps)
    out += lam * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def rank_aware_scaling(probs, n_windows=N_WINDOWS, power=0.4):
    view     = probs.reshape(-1, n_windows, probs.shape[1])
    file_max = view.max(axis=1, keepdims=True)
    return (view * np.power(file_max, power)).reshape(probs.shape)

def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.20):
    result = probs.copy(); view = probs.reshape(-1, n_windows, probs.shape[1])
    out    = result.reshape(-1, n_windows, probs.shape[1])
    for t in range(n_windows):
        conf  = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        if t == 0:
            nbr = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            nbr = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

TEXTURE_TAXA  = {'Amphibia', 'Insecta'}
temperatures  = np.ones(N_CLASSES, dtype=np.float32)
for ci, lbl in enumerate(PRIMARY_LABELS):
    temperatures[ci] = 0.95 if CLASS_NAME_MAP.get(lbl,'Aves') in TEXTURE_TAXA else 1.10

print('Post-processing helpers ready')

Post-processing helpers ready


In [8]:
# MLP probes — upgraded (pca_dim=128, richer features, bigger hidden)

def seq_features_1d(col, n_windows=N_WINDOWS):
    x     = col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    nxt   = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean  = np.repeat(x.mean(1), n_windows)
    mx    = np.repeat(x.max(1),  n_windows)
    std   = np.repeat(x.std(1),  n_windows)
    return prev, nxt, mean, mx, std

def build_class_features(Z, raw_col, prior_col):
    prev, nxt, mean, mx, std = seq_features_1d(prior_col)
    diff_mean = prior_col - mean
    diff_prev = prior_col - prev
    diff_next = prior_col - nxt
    return np.hstack([
        Z,
        raw_col[:, None], prior_col[:, None],
        prev[:, None], nxt[:, None], mean[:, None], mx[:, None], std[:, None],
        diff_mean[:, None], diff_prev[:, None], diff_next[:, None],
        (raw_col * prior_col)[:, None],
        np.abs(raw_col - prior_col)[:, None],
    ])

def train_mlp_probes(emb, scores_raw, scores_prior, Y, cfg):
    scaler = StandardScaler()
    Z      = PCA(n_components=min(cfg['pca_dim'], emb.shape[0]-1))\
             .fit_transform(scaler.fit_transform(emb)).astype(np.float32)
    print(f'  PCA: {emb.shape} → {Z.shape}')

    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / Y.shape[0]
    cw        = np.clip(1.0 / (freq**0.5), 1.0, 10.0)
    cw        /= cw.mean()

    probe_models = {}; MAX_ROWS = 2500
    active = np.where(Y.sum(0) >= cfg['min_pos'])[0]
    print(f'  Training {len(active)} MLP probes...')

    for ci in active:
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue
        X = build_class_features(Z, scores_raw[:, ci], scores_prior[:, ci])
        n_pos = int(y.sum()); pos_idx = np.where(y == 1)[0]
        repeat = min(max(1, int(round(cw[ci] * (len(y)-n_pos)/max(n_pos,1)))), 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_b = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_b = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(
            hidden_layer_sizes=cfg['hidden'], activation='relu',
            max_iter=200, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=15,
            random_state=42, learning_rate_init=5e-4, alpha=0.005,
        )
        clf.fit(X_b, y_b)
        probe_models[ci] = clf

    print(f'  Trained {len(probe_models)} probes')
    pca_obj = PCA(n_components=min(cfg['pca_dim'], emb.shape[0]-1))
    pca_obj.fit(scaler.transform(emb))
    return probe_models, scaler, pca_obj, cfg['alpha_blend']

def apply_mlp_probes(emb_test, scores_raw, scores_prior, probe_models, scaler, pca_obj, alpha_blend):
    Z_test = pca_obj.transform(scaler.transform(emb_test)).astype(np.float32)
    result = scores_raw.copy()
    for ci, clf in probe_models.items():
        X_test = build_class_features(Z_test, scores_raw[:, ci], scores_prior[:, ci])
        prob   = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit  = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_raw[:, ci] + alpha_blend * logit
    return result

print('MLP probes ready')

MLP probes ready


In [9]:
# ProtoSSM v4 — larger model with TemporalCrossAttention + Focal + Mixup + SWA

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, 2*d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D     = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        xz = self.in_proj(x); x_ssm, z = xz.chunk(2, dim=-1)
        x_c = self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2)
        x_c = F.silu(x_c)
        dt  = F.softplus(self.dt_proj(x_c))
        A   = -torch.exp(self.A_log)
        B_  = self.B_proj(x_c); C = self.C_proj(x_c)
        h   = torch.zeros(B, D, self.d_state, device=x.device)
        ys  = []
        for t in range(T):
            dt_t = dt[:, t, :]
            h = h * torch.exp(A[None] * dt_t[:, :, None]) + x[:, t, :, None] * (dt_t[:, :, None] * B_[:, t, None, :])
            ys.append((h * C[:, t, None, :]).sum(-1))
        return torch.stack(ys, dim=1) + x * self.D[None, None, :]


class TemporalCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model), nn.Dropout(dropout))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        res = x; x = self.norm(x)
        attn_out, _ = self.attn(x, x, x)
        x = res + attn_out
        res = x; x = self.norm2(x)
        return res + self.ffn(x)


class ProtoSSMv4(nn.Module):
    def __init__(self, d_input=1536, d_model=256, d_state=16, n_ssm_layers=3,
                 n_classes=234, n_windows=N_WINDOWS, dropout=0.12,
                 n_sites=20, meta_dim=24, use_cross_attn=True, cross_attn_heads=8,
                 n_families=0):
        super().__init__()
        self.n_classes = n_classes; self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn; self.n_families = n_families

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc    = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb   = nn.Embedding(n_sites, meta_dim)
        self.hour_emb   = nn.Embedding(24, meta_dim)
        self.meta_proj  = nn.Linear(2*meta_dim, d_model)

        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2*d_model, d_model) for _ in range(n_ssm_layers)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.drop = nn.Dropout(dropout)

        if use_cross_attn:
            self.cross_attn = TemporalCrossAttention(d_model, n_heads=cross_attn_heads, dropout=dropout)

        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

        if n_families > 0:
            self.family_head = nn.Linear(d_model, n_families)
        else:
            self.family_head = None

    def init_prototypes(self, emb, labels):
        with torch.no_grad():
            h = self.input_proj(emb)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            site_ids = site_ids.clamp(0, self.site_emb.num_embeddings-1)
            hours    = hours.clamp(0, 23)
            meta = self.meta_proj(torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]
        for fwd, bwd, merge, norm in zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
        if self.use_cross_attn:
            h = self.cross_attn(h)
        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp) + self.class_bias[None,None,:]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            out   = alpha * sim + (1.0 - alpha) * perch_logits
        else:
            out = sim
        fam_out = None
        if self.family_head is not None:
            fam_out = self.family_head(h.mean(dim=1))
        return out, fam_out

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def mixup_files(emb, logits, labels, alpha=0.4):
    n   = len(emb)
    if alpha <= 0 or n < 2: return emb, logits, labels
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)
    perm = np.random.permutation(n)
    return (lam*emb + (1-lam)*emb[perm],
            lam*logits + (1-lam)*logits[perm],
            lam*labels + (1-lam)*labels[perm])


def train_proto_ssm_v4(emb_full, scores_full, Y_full, meta_full,
                        family_labels=None, cfg=None, n_sites=20, verbose=False):
    if cfg is None: cfg = CFG['proto_ssm_train']
    n_epochs      = cfg['n_epochs']
    label_sm      = cfg.get('label_smoothing', 0.03)
    mixup_alpha   = cfg.get('mixup_alpha', 0.4)
    focal_gamma   = cfg.get('focal_gamma', 2.5)
    swa_start     = int(n_epochs * cfg.get('swa_start_frac', 0.65))
    patience      = cfg.get('patience', 8)

    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f   = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)
    if label_sm > 0:
        lab_f = lab_f * (1.0 - label_sm) + label_sm / 2.0

    fnames  = meta_full.groupby('filename', sort=False).size().index.tolist()
    sites_u = sorted(meta_full['site'].astype(str).unique())
    site2i  = {s: i+1 for i, s in enumerate(sites_u)}
    site_ids = np.array([min(site2i.get(str(meta_full.loc[meta_full['filename']==fn,'site'].iloc[0]),0), n_sites-1) for fn in fnames], dtype=np.int64)
    hour_ids = np.array([int(meta_full.loc[meta_full['filename']==fn,'hour_utc'].iloc[0]) % 24 for fn in fnames], dtype=np.int64)

    n_fam = 0
    if family_labels is not None:
        n_fam = family_labels.shape[1]

    ssm_cfg = CFG['proto_ssm']
    model = ProtoSSMv4(
        d_model=ssm_cfg['d_model'], d_state=ssm_cfg['d_state'],
        n_ssm_layers=ssm_cfg['n_ssm_layers'],
        n_classes=N_CLASSES, n_windows=N_WINDOWS,
        dropout=ssm_cfg['dropout'], n_sites=n_sites,
        meta_dim=ssm_cfg['meta_dim'],
        use_cross_attn=ssm_cfg['use_cross_attn'],
        cross_attn_heads=ssm_cfg['cross_attn_heads'],
        n_families=n_fam,
    )
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full, dtype=torch.float32))
    if verbose: print(f'  ProtoSSMv4 params: {model.count_parameters():,}')

    pos_cnt    = torch.tensor(lab_f, dtype=torch.float32).sum(dim=(0,1))
    pos_weight = ((n_files * N_WINDOWS - pos_cnt) / (pos_cnt + 1)).clamp(max=cfg['pos_weight_cap'])

    opt   = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg['lr'], epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy='cos')

    best_loss  = float('inf'); best_state = None; wait = 0
    swa_state  = None; swa_count = 0

    for ep in range(n_epochs):
        if mixup_alpha > 0 and ep > 5:
            e_m, l_m, y_m = mixup_files(emb_f, log_f, lab_f, alpha=mixup_alpha)
        else:
            e_m, l_m, y_m = emb_f, log_f, lab_f

        model.train()
        out, fam_out = model(
            torch.tensor(e_m, dtype=torch.float32),
            torch.tensor(l_m, dtype=torch.float32),
            site_ids=torch.tensor(site_ids, dtype=torch.long),
            hours=torch.tensor(hour_ids, dtype=torch.long),
        )
        y_t = torch.tensor(y_m, dtype=torch.float32)
        if focal_gamma > 0:
            loss = focal_bce(out, y_t, gamma=focal_gamma, pos_weight=pos_weight[None,None,:])
        else:
            loss = F.binary_cross_entropy_with_logits(out, y_t, pos_weight=pos_weight[None,None,:])
        loss = loss + cfg['distill_weight'] * F.mse_loss(out, torch.tensor(l_m, dtype=torch.float32))
        if fam_out is not None and family_labels is not None:
            fam_t = torch.tensor(family_labels, dtype=torch.float32)
            loss  = loss + 0.1 * F.binary_cross_entropy_with_logits(fam_out, fam_t)

        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        if ep >= swa_start:
            if swa_state is None:
                swa_state = {k: v.clone() for k, v in model.state_dict().items()}; swa_count = 1
            else:
                for k in swa_state: swa_state[k] += model.state_dict()[k]
                swa_count += 1

        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
        if wait >= patience: break

    if swa_state is not None and swa_count >= 3:
        model.load_state_dict({k: v / swa_count for k, v in swa_state.items()})
    elif best_state:
        model.load_state_dict(best_state)
    model.eval()
    if verbose: print(f'  best_loss={best_loss:.4f}  swa_count={swa_count}')
    return model, site2i

print('ProtoSSM v4 ready')

ProtoSSM v4 ready


In [10]:
# Taxonomy groups for auxiliary head
def build_taxonomy_groups(taxonomy_df, primary_labels):
    grp_col   = 'class_name' if 'class_name' in taxonomy_df.columns else None
    if grp_col is None: return 1, [0]*len(primary_labels), {}
    grp_map   = taxonomy_df.set_index('primary_label')[grp_col].to_dict()
    groups    = sorted(set(grp_map.values()))
    grp_to_i  = {g: i for i, g in enumerate(groups)}
    class_grp = [grp_to_i.get(grp_map.get(lbl,'Unknown'), 0) for lbl in primary_labels]
    return len(groups), class_grp, grp_to_i

n_families, class_to_family, fam_to_idx = build_taxonomy_groups(taxonomy, PRIMARY_LABELS)
print(f'Taxonomy groups: {n_families}  ({list(fam_to_idx.keys())})')

# Build per-file family multi-hot labels
n_files_all    = len(sc_tr) // N_WINDOWS
emb_files_all  = emb_tr.reshape(n_files_all, N_WINDOWS, -1)
lab_files_all  = Y_FULL_aligned.reshape(n_files_all, N_WINDOWS, -1)
file_families  = np.zeros((n_files_all, n_families), dtype=np.float32)
for fi in range(n_files_all):
    for ci in np.where(lab_files_all[fi].sum(axis=0) > 0)[0]:
        file_families[fi, class_to_family[ci]] = 1.0

print(f'Family labels shape: {file_families.shape}')

Taxonomy groups: 5  (['Amphibia', 'Aves', 'Insecta', 'Mammalia', 'Reptilia'])
Family labels shape: (59, 5)


In [11]:
# Full OOF pipeline
prior_tables = build_prior_tables(sc, Y_SC)

# Prior-adjusted raw scores (used as extra features for MLP)
sc_tr_prior = apply_prior(sc_tr,
    sites=meta_tr['site'].to_numpy(),
    hours=meta_tr['hour_utc'].to_numpy(),
    tables=prior_tables, lam=CFG['lambda_prior'])

file_meta = meta_tr.drop_duplicates('filename').reset_index(drop=True)
gkf       = GroupKFold(n_splits=N_OOF_SPLITS)
oof_probs = np.zeros((len(sc_tr), N_CLASSES), dtype=np.float32)
fold_aucs = []

for fold, (tr_f, va_f) in enumerate(gkf.split(file_meta, groups=file_meta['filename']), 1):
    t0 = time.time()
    tr_fnames = set(file_meta.iloc[tr_f]['filename'])
    va_fnames = set(file_meta.iloc[va_f]['filename'])
    tr_mask   = meta_tr['filename'].isin(tr_fnames).values
    va_mask   = meta_tr['filename'].isin(va_fnames).values

    # Fold-safe prior
    sc_clean_tr = sc[sc['filename'].isin(tr_fnames)]
    Y_clean_tr  = Y_SC[sc_clean_tr.index]
    prior_fold  = build_prior_tables(sc_clean_tr.reset_index(drop=True), Y_clean_tr)
    sc_va_prior = apply_prior(sc_tr[va_mask],
        sites=meta_tr[va_mask]['site'].to_numpy(),
        hours=meta_tr[va_mask]['hour_utc'].to_numpy(),
        tables=prior_fold, lam=CFG['lambda_prior'])
    sc_tr_prior_fold = apply_prior(sc_tr[tr_mask],
        sites=meta_tr[tr_mask]['site'].to_numpy(),
        hours=meta_tr[tr_mask]['hour_utc'].to_numpy(),
        tables=prior_fold, lam=CFG['lambda_prior'])

    print(f'\nFold {fold}/{N_OOF_SPLITS} — train: {len(tr_fnames)} files, val: {len(va_fnames)} files')

    # File-level family labels for train fold
    tr_file_idx = [i for i, fn in enumerate(file_meta['filename']) if fn in tr_fnames]
    fam_tr = file_families[tr_file_idx]

    # ProtoSSM v4
    proto_model, site2i = train_proto_ssm_v4(
        emb_tr[tr_mask], sc_tr[tr_mask], Y_FULL_aligned[tr_mask],
        meta_tr[tr_mask].reset_index(drop=True),
        family_labels=fam_tr, verbose=True)

    n_va = int(va_mask.sum()) // N_WINDOWS
    va_fnames_list = meta_tr[va_mask].drop_duplicates('filename')['filename'].tolist()
    va_site_ids = np.array([min(site2i.get(meta_tr.loc[meta_tr['filename']==fn,'site'].iloc[0],0), 19)
                             for fn in va_fnames_list], dtype=np.int64)
    va_hour_ids = np.array([int(meta_tr.loc[meta_tr['filename']==fn,'hour_utc'].iloc[0])%24
                             for fn in va_fnames_list], dtype=np.int64)

    with torch.no_grad():
        proto_va, _ = proto_model(
            torch.tensor(emb_tr[va_mask].reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
            torch.tensor(sc_tr[va_mask].reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
            site_ids=torch.tensor(va_site_ids, dtype=torch.long),
            hours=torch.tensor(va_hour_ids, dtype=torch.long))
    proto_va = proto_va.numpy().reshape(-1, N_CLASSES)

    # MLP probes
    probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
        emb_tr[tr_mask], sc_tr[tr_mask], sc_tr_prior_fold,
        Y_FULL_aligned[tr_mask], CFG['mlp'])
    sc_va_mlp = apply_mlp_probes(
        emb_tr[va_mask], sc_tr[va_mask], sc_va_prior,
        probe_models, emb_scaler, emb_pca, alpha_blend)

    # Ensemble: 25% ProtoSSM + 50% MLP + 25% prior
    first_pass = 0.25 * proto_va + 0.50 * sc_va_mlp + 0.25 * sc_va_prior
    probs_va   = sigmoid(first_pass / temperatures[None, :])

    # Post-processing
    probs_va = rank_aware_scaling(probs_va, power=CFG['rank_aware_power'])
    probs_va = adaptive_delta_smooth(probs_va, base_alpha=CFG['delta_smooth_alpha'])
    probs_va = np.clip(probs_va, 0.0, 1.0)

    oof_probs[va_mask] = probs_va
    Y_va = Y_FULL_aligned[va_mask]
    fold_auc = macro_auc(Y_va, probs_va)
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold} AUC={fold_auc:.5f}  ({time.time()-t0:.0f}s)')

print(f'\nMean fold AUC: {np.mean(fold_aucs):.5f}')


Fold 1/3 — train: 39 files, val: 20 files
  ProtoSSMv4 params: 2,659,066


  best_loss=0.6412  swa_count=7
  PCA: (468, 1536) → (468, 128)
  Training 45 MLP probes...


  Trained 45 probes
  Fold 1 AUC=0.89505  (39s)

Fold 2/3 — train: 39 files, val: 20 files
  ProtoSSMv4 params: 2,659,066


  best_loss=0.6638  swa_count=7
  PCA: (468, 1536) → (468, 128)
  Training 53 MLP probes...


  Trained 53 probes
  Fold 2 AUC=0.92375  (38s)

Fold 3/3 — train: 40 files, val: 19 files
  ProtoSSMv4 params: 2,659,066


  best_loss=0.6493  swa_count=7
  PCA: (480, 1536) → (480, 128)
  Training 51 MLP probes...


  Trained 51 probes
  Fold 3 AUC=0.90657  (39s)

Mean fold AUC: 0.90846


In [12]:
# Final scores
oof_auc  = macro_auc(Y_FULL_aligned, oof_probs)
oof_cmap = padded_cmap(Y_FULL_aligned, oof_probs)

raw_probs = sigmoid(sc_tr / temperatures[None, :])
raw_auc   = macro_auc(Y_FULL_aligned, raw_probs)
raw_cmap  = padded_cmap(Y_FULL_aligned, raw_probs)

print('=' * 60)
print(f'Raw Perch     OOF AUC={raw_auc:.5f}  cmAP={raw_cmap:.5f}')
print(f'Full pipeline OOF AUC={oof_auc:.5f}  cmAP={oof_cmap:.5f}')
print(f'Delta AUC: {oof_auc-raw_auc:+.5f}  Delta cmAP: {oof_cmap-raw_cmap:+.5f}')
print(f'Total wall time: {(time.time()-_WALL_START)/60:.1f} min')

Raw Perch     OOF AUC=0.73902  cmAP=0.86916
Full pipeline OOF AUC=0.90468  cmAP=0.90855
Delta AUC: +0.16566  Delta cmAP: +0.03939
Total wall time: 2.0 min
